In [1]:
# ============================================================
# LLAMAINDEX RAG DEMONSTRATION — SINGLE JUPYTER NOTEBOOK CELL
# ============================================================

# ------------------------------------------------------------
# 1. Install the required packages
# ------------------------------------------------------------

import sys
import subprocess
import os
from getpass import getpass

packages = [
    "llama-index",
    "llama-index-llms-openai",
    "llama-index-embeddings-openai"
]

print("Installing the required LlamaIndex packages...")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        *packages
    ]
)

print("Packages installed successfully.\n")



Installing the required LlamaIndex packages...
Packages installed successfully.



In [2]:
# ------------------------------------------------------------
# 2. Import LlamaIndex components
# ------------------------------------------------------------

from llama_index.core import (
    Document,
    Settings,
    VectorStoreIndex
)

from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding


In [3]:
from google.colab import userdata
from dotenv import load_dotenv
load_dotenv()

# Get the API key from environment variables
openai_api_key = userdata.get("OPENAI_API_KEY")
import os
os.environ["OPENAI_API_KEY"] = openai_api_key

# ------------------------------------------------------------
# 3. Read the OpenAI API key securely
# ------------------------------------------------------------

#if not os.environ.get("OPENAI_API_KEY"):
#    os.environ["OPENAI_API_KEY"] = getpass(
#        "Enter your OpenAI API key: "
#    )


In [4]:
# ------------------------------------------------------------
# 4. Configure the LLM and embedding model
# ------------------------------------------------------------

Settings.llm = OpenAI(
    model="gpt-4o-mini",
    temperature=0
)

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small"
)

# Configure document chunking
Settings.chunk_size = 256
Settings.chunk_overlap = 30

In [5]:
# ------------------------------------------------------------
# 5. Create sample knowledge-base documents
# ------------------------------------------------------------

knowledge_base = [
    """
    ABC Internet Services: Router Troubleshooting

    If the router's red light is blinking, restart the router
    and verify that the fibre-optic cable is securely connected.

    Switch off the router, wait for approximately thirty seconds,
    and then switch it on again.

    If the red light continues blinking after the restart,
    contact the technical-support department.
    """,

    """
    ABC Internet Services: Slow Internet Connection

    If the internet connection is slow, first check the Wi-Fi
    signal strength.

    Move closer to the router and disconnect unnecessary devices.

    Restart the modem and router.

    Customers should also check whether large downloads or
    software updates are consuming the available bandwidth.
    """,

    """
    ABC Internet Services: Password Management

    Customers can reset their account password by selecting
    Forgot Password on the login page.

    A password-reset link will be sent to the customer's
    registered email address.

    The new password should contain at least eight characters,
    one uppercase letter, one lowercase letter and one number.
    """,

    """
    ABC Internet Services: Warranty Policy

    Warranty replacement is available within one year from
    the original purchase date.

    The customer must provide the purchase invoice and the
    serial number of the device.

    Physical damage, water damage and damage caused by
    unauthorized repairs are not covered by the warranty.
    """,

    """
    ABC Internet Services: Billing Support

    For billing problems, customers should contact the accounts
    department and provide their invoice number.

    Billing complaints may include incorrect charges, duplicate
    payments, failed transactions and missing payment receipts.

    The accounts department normally responds within two
    working days.
    """,

    """
    ABC Internet Services: Support Hours

    Technical support is available from Monday to Saturday,
    between 9:00 a.m. and 6:00 p.m.

    Billing support is available from Monday to Friday,
    between 10:00 a.m. and 5:00 p.m.

    Emergency network-outage complaints can be registered
    through the customer-support portal at any time.
    """
]


# Convert the text into LlamaIndex Document objects
documents = []

for document_number, text in enumerate(
    knowledge_base,
    start=1
):
    document = Document(
        text=text.strip(),
        metadata={
            "document_number": document_number,
            "source": "ABC Internet Services Knowledge Base"
        }
    )

    documents.append(document)

print(f"Created {len(documents)} knowledge-base documents.")

Created 6 knowledge-base documents.


In [6]:
# ------------------------------------------------------------
# 6. Create the vector index
# ------------------------------------------------------------

print("Creating embeddings and building the vector index...")

index = VectorStoreIndex.from_documents(
    documents,
    show_progress=True
)

print("Vector index created successfully.\n")


# ------------------------------------------------------------
# 7. Create the query engine
# ------------------------------------------------------------

query_engine = index.as_query_engine(
    similarity_top_k=3,
    response_mode="compact"
)


# ------------------------------------------------------------
# 8. Ask a demonstration question
# ------------------------------------------------------------

demo_question = (
    "What should I do if the router red light is blinking?"
)

print("=" * 75)
print("DEMONSTRATION QUESTION")
print("=" * 75)
print(demo_question)

demo_response = query_engine.query(demo_question)

print("\nANSWER")
print("-" * 75)
print(demo_response)


Creating embeddings and building the vector index...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/6 [00:00<?, ?it/s]

Vector index created successfully.

DEMONSTRATION QUESTION
What should I do if the router red light is blinking?

ANSWER
---------------------------------------------------------------------------
Restart the router and ensure that the fibre-optic cable is securely connected. Switch off the router, wait for approximately thirty seconds, and then turn it back on. If the red light continues to blink after the restart, contact the technical-support department.


In [7]:
# ------------------------------------------------------------
# 9. Display the retrieved source chunks
# ------------------------------------------------------------

print("\nRETRIEVED SOURCES")
print("-" * 75)

for source_number, source_node in enumerate(
    demo_response.source_nodes,
    start=1
):
    score = source_node.score

    if score is None:
        score_text = "Not available"
    else:
        score_text = f"{score:.4f}"

    metadata = source_node.node.metadata
    source_text = source_node.node.get_content().strip()

    print(f"\nSource {source_number}")
    print(f"Similarity score: {score_text}")
    print(
        "Document number:",
        metadata.get("document_number", "Unknown")
    )
    print("Retrieved text:")
    print(source_text[:600])



RETRIEVED SOURCES
---------------------------------------------------------------------------

Source 1
Similarity score: 0.7172
Document number: 1
Retrieved text:
ABC Internet Services: Router Troubleshooting

    If the router's red light is blinking, restart the router
    and verify that the fibre-optic cable is securely connected.

    Switch off the router, wait for approximately thirty seconds,
    and then switch it on again.

    If the red light continues blinking after the restart,
    contact the technical-support department.

Source 2
Similarity score: 0.3621
Document number: 2
Retrieved text:
ABC Internet Services: Slow Internet Connection

    If the internet connection is slow, first check the Wi-Fi
    signal strength.

    Move closer to the router and disconnect unnecessary devices.

    Restart the modem and router.

    Customers should also check whether large downloads or
    software updates are consuming the available bandwidth.

Source 3
Similarity score: 0.2

In [8]:
# ------------------------------------------------------------
# 10. Start the interactive question-answering application
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("LLAMAINDEX CUSTOMER-SUPPORT RAG APPLICATION")
print("=" * 75)

print(
    "\nAsk questions about router problems, internet speed, "
    "passwords, warranty, billing or support hours."
)

print("Type 'exit' to stop the application.\n")


while True:

    question = input("Your question: ").strip()

    if question.lower() in {
        "exit",
        "quit",
        "stop"
    }:
        print("\nLlamaIndex application stopped.")
        break

    if not question:
        print("Please enter a valid question.\n")
        continue

    try:
        response = query_engine.query(question)

        print("\nANSWER")
        print("-" * 75)
        print(response)

        print("\nRETRIEVED SOURCES")
        print("-" * 75)

        if not response.source_nodes:
            print("No source nodes were returned.")

        for source_number, source_node in enumerate(
            response.source_nodes,
            start=1
        ):
            score = source_node.score

            if score is None:
                score_text = "Not available"
            else:
                score_text = f"{score:.4f}"

            metadata = source_node.node.metadata

            source_text = (
                source_node.node
                .get_content()
                .strip()
                .replace("\n", " ")
            )

            print(
                f"\nSource {source_number} "
                f"— similarity score: {score_text}"
            )

            print(
                "Document number:",
                metadata.get(
                    "document_number",
                    "Unknown"
                )
            )

            print(source_text[:400])

        print("\n" + "=" * 75 + "\n")

    except Exception as error:
        print("\nThe query could not be completed.")
        print(
            f"{type(error).__name__}: {error}\n"
        )


LLAMAINDEX CUSTOMER-SUPPORT RAG APPLICATION

Ask questions about router problems, internet speed, passwords, warranty, billing or support hours.
Type 'exit' to stop the application.

Your question: What should I do if the internet speed is low?

ANSWER
---------------------------------------------------------------------------
To address slow internet speed, first check the Wi-Fi signal strength. Try moving closer to the router and disconnect any unnecessary devices. Restart both the modem and the router. Additionally, ensure that large downloads or software updates are not consuming your available bandwidth.

RETRIEVED SOURCES
---------------------------------------------------------------------------

Source 1 — similarity score: 0.5879
Document number: 2
ABC Internet Services: Slow Internet Connection      If the internet connection is slow, first check the Wi-Fi     signal strength.      Move closer to the router and disconnect unnecessary devices.      Restart the modem and route